# Machine Learning Models: 30-Day Readmission Prediction

## Objective
This notebook builds and compares four strong baseline-to-advanced machine learning models for predicting **30-day hospital readmission** using the cleaned Diabetes 130-US Hospitals dataset. The goal is to identify the top candidates for your end-to-end project and later select **2 models for deployment**.

## Why these models
EDA shows strong class imbalance (about 11.2% positive class), prior utilisation as the strongest signal, and useful contributions from discharge disposition, medical specialty, diagnosis context, insulin, medication change, and clinical complexity. That means the modelling workflow must handle mixed numeric/categorical inputs, imbalance, and nonlinear relationships.

## Modelling strategy
We do not jump directly to the most complex model. First, we validate modelling conditions, prepare a leakage-safe feature set, split the data correctly, and evaluate models using metrics that fit the professor's requirement and the business objective.

## Top 4 candidate models
1. **Logistic Regression**  -  interpretable baseline, useful for academic explanation and deployment.
2. **Random Forest**  -  robust nonlinear ensemble, handles interactions well after encoding.
3. **XGBoost**  -  high-performing gradient boosting model, often strongest on tabular healthcare data.
4. **LightGBM**  -  efficient boosting model, strong on wide encoded datasets and fast to tune.

## Evaluation rule
Because the dataset is imbalanced, **accuracy alone is misleading**. We compare models primarily using Recall, F1, ROC-AUC, and PR-AUC, while still reporting precision and accuracy for completeness. Threshold tuning can be added later before deployment.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report
)

try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False

try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except Exception:
    lgbm_available = False

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


## Step 1: Load data and check modelling conditions
Before training models, we confirm that the target exists, leakage-prone fields are excluded, categorical/numeric columns are identified properly, and logically impossible readmission discharge groups are removed from the modelling sample.


In [3]:
DATA_PATHS = [
    Path('cleaned_data.csv'),
    Path('data/processed/cleaned_data.csv'),
    Path('../data/processed/cleaned_data.csv'),
]

for path in DATA_PATHS:
    if path.exists():
        df = pd.read_csv(path)
        data_path_used = path
        break
else:
    raise FileNotFoundError('cleaned_data.csv not found in expected locations.')

# Rename columns to match the expected names in the notebook
df = df.rename(columns={
    'time_in_hospital': 'timeinhospital',
    'number_inpatient': 'numberinpatient',
    'discharge_disposition': 'dischargedisposition',
    'readmitted_binary': 'readmittedbinary'
})

print(f'Dataset loaded from: {data_path_used}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Readmitted within 30 days: {df["readmittedbinary"].sum():,} ({df["readmittedbinary"].mean()*100:.2f}%)')
df.head(3)

Dataset loaded from: ..\data\processed\cleaned_data.csv
Shape: 101,766 rows x 47 columns
Readmitted within 30 days: 11,357 (11.16%)


,race,gender,timeinhospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,numberinpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmittedbinary,age_numeric,admission_type,dischargedisposition,admission_source
0,Caucasian,Female,1,Unknown,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,Unknown,Unknown,1,Unknown,Unknown,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0,5.0,NaN,Not Mapped,Physician Referral
1,Caucasian,Female,3,Unknown,Unknown,59,0,18,0,0,0,276,250.01,255,9,Unknown,Unknown,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0,15.0,Emergency,Discharged to home,Emergency Room
2,AfricanAmerican,Female,2,Unknown,Unknown,11,5,13,2,0,1,648,250,V27,6,Unknown,Unknown,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0,25.0,Emergency,Discharged to home,Emergency Room


In [4]:
impossible_keywords = ['Expired', 'Hospice', 'hospice', 'expired']
impossible_mask = df['dischargedisposition'].astype(str).str.contains('|'.join(impossible_keywords), case=False, na=False)
analysis_df = df.loc[~impossible_mask].copy()

print(f'Rows removed from impossible readmission discharge groups: {int(impossible_mask.sum()):,}')
print(f'Analysis sample shape: {analysis_df.shape[0]:,} rows x {analysis_df.shape[1]} columns')
print(f'Analysis readmission rate: {analysis_df["readmittedbinary"].mean()*100:.2f}%')


Rows removed from impossible readmission discharge groups: 2,423
Analysis sample shape: 99,343 rows x 47 columns
Analysis readmission rate: 11.39%


## Step 2: Define leakage-safe features
The cleaning notebook already removed `encounterid`, `patientnbr`, and original coded replacements, which helps prevent direct leakage. Here we also keep the modelling target separate and use all remaining cleaned predictors.


In [5]:
target_col = 'readmittedbinary'
drop_cols = [target_col]
X = analysis_df.drop(columns=drop_cols).copy()
y = analysis_df[target_col].copy()

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

print(f'Total features: {X.shape[1]}')
print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')
print('Sample numeric features:', numeric_features[:10])
print('Sample categorical features:', categorical_features[:10])


Total features: 46
Numeric features: 9
Categorical features: 37
Sample numeric features: ['timeinhospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'numberinpatient', 'number_diagnoses', 'age_numeric']
Sample categorical features: ['race', 'gender', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin']


## Step 3: Train-test split
We use a stratified split so the positive class proportion stays similar in both train and test sets. This is important because the readmission class is rare.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print(f'Train positive rate: {y_train.mean()*100:.2f}%')
print(f'Test positive rate: {y_test.mean()*100:.2f}%')


Train shape: (79474, 46)
Test shape: (19869, 46)
Train positive rate: 11.39%
Test positive rate: 11.39%


## Step 4: Preprocessing pipeline
- Numeric columns: median imputation, then scaling for models that benefit from it.
- Categorical columns: most-frequent imputation, then one-hot encoding with unknown-category handling.
This lets one pipeline serve all four models consistently.


In [7]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


## Step 5: Select the top 4 models
These four models are chosen because together they give strong academic coverage, interpretability, and tabular-data performance.

### Why these are suitable
- Logistic Regression: best for explanation, coefficient-based reasoning, and lightweight deployment.
- Random Forest: handles nonlinearities and interactions better than linear models.
- XGBoost: usually one of the strongest choices for structured data.
- LightGBM: similar strength to XGBoost but often faster and more efficient with wide encoded features.


In [8]:
models = {}

models['Logistic Regression'] = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

models['Random Forest'] = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_split=10, min_samples_leaf=5,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ))
])

if xgb_available:
    models['XGBoost'] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, eval_metric='logloss', random_state=42,
            scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1])
        ))
    ])
else:
    print('XGBoost not available in environment.')

if lgbm_available:
    models['LightGBM'] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            class_weight='balanced', random_state=42, verbose=-1
        ))
    ])
else:
    print('LightGBM not available in environment.')

list(models.keys())


['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']

## Step 6: Train and evaluate all models
For each model, we compute accuracy, precision, recall, F1, ROC-AUC, and PR-AUC. Because the positive class is limited, Recall and PR-AUC are especially important for deciding which models deserve deployment.


In [9]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
        'PR_AUC': average_precision_score(y_test, y_prob) if y_prob is not None else np.nan
    }

    cm = confusion_matrix(y_test, y_pred)
    return model, metrics, cm, classification_report(y_test, y_pred, zero_division=0)

trained_models = {}
results = []
conf_matrices = {}
reports = {}

for name, model in models.items():
    trained_model, metrics, cm, report = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    trained_models[name] = trained_model
    results.append(metrics)
    conf_matrices[name] = cm
    reports[name] = report
    print(f'Finished training: {name}')


Finished training: Logistic Regression
Finished training: Random Forest
Finished training: XGBoost
Finished training: LightGBM


## Step 7: Model comparison table
This table is the key deliverable for the professor because it compares the four best candidate models side by side.


In [10]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['PR_AUC', 'Recall', 'ROC_AUC', 'F1'], ascending=False).reset_index(drop=True)
results_df.round(4)


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,LightGBM,0.6813,0.1884,0.5435,0.2798,0.6745,0.2403
1,XGBoost,0.6865,0.1921,0.5466,0.2843,0.6804,0.2403
2,Random Forest,0.7598,0.2175,0.4269,0.2881,0.6781,0.2265
3,Logistic Regression,0.6644,0.1794,0.5444,0.2698,0.6555,0.2155


## Step 8: Rank the best 4 and recommend the top 2 for deployment
The notebook first keeps all four as finalists, then gives a deployment recommendation based on imbalance-aware performance. For a healthcare readmission problem, strong Recall and PR-AUC usually matter more than raw Accuracy.


In [11]:
results_df['Rank'] = range(1, len(results_df) + 1)
display(results_df[['Rank', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'PR_AUC']].round(4))

deployment_candidates = results_df.head(2)['Model'].tolist()
print('Recommended top 2 models for deployment:')
for i, m in enumerate(deployment_candidates, 1):
    print(f'{i}. {m}')


,Rank,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,1,LightGBM,0.6813,0.1884,0.5435,0.2798,0.6745,0.2403
1,2,XGBoost,0.6865,0.1921,0.5466,0.2843,0.6804,0.2403
2,3,Random Forest,0.7598,0.2175,0.4269,0.2881,0.6781,0.2265
3,4,Logistic Regression,0.6644,0.1794,0.5444,0.2698,0.6555,0.2155


Recommended top 2 models for deployment:
1. LightGBM
2. XGBoost


## Step 9: Confusion matrices and detailed classification reports
These outputs help the group explain trade-offs during the practical demonstration, especially false negatives versus false positives.


In [12]:
for name in results_df['Model']:
    print('=' * 80)
    print(name)
    print('Confusion Matrix:')
    print(conf_matrices[name])
    print('Classification Report:')
    print(reports[name])


LightGBM
Confusion Matrix:
[[12307  5299]
 [ 1033  1230]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.70      0.80     17606
           1       0.19      0.54      0.28      2263

    accuracy                           0.68     19869
   macro avg       0.56      0.62      0.54     19869
weighted avg       0.84      0.68      0.74     19869

XGBoost
Confusion Matrix:
[[12404  5202]
 [ 1026  1237]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.70      0.80     17606
           1       0.19      0.55      0.28      2263

    accuracy                           0.69     19869
   macro avg       0.56      0.63      0.54     19869
weighted avg       0.84      0.69      0.74     19869

Random Forest
Confusion Matrix:
[[14130  3476]
 [ 1297   966]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.80      0.86 

## Step 10: Feature importance / explainability
For presentation, at least one interpretable model is useful. Logistic Regression gives coefficient-based insight, and tree models provide feature importance. This helps each group member discuss the hypothesized drivers of readmission.


In [13]:
best_model_name = results_df.loc[0, 'Model']
best_model = trained_models[best_model_name]
print('Best-performing model in this run:', best_model_name)

if best_model_name == 'Logistic Regression':
    feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
    coefs = best_model.named_steps['model'].coef_[0]
    importance_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
    importance_df['AbsCoefficient'] = importance_df['Coefficient'].abs()
    importance_df = importance_df.sort_values('AbsCoefficient', ascending=False).head(20)
    display(importance_df[['Feature', 'Coefficient']].round(4))
elif best_model_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
    importances = best_model.named_steps['model'].feature_importances_
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    importance_df = importance_df.sort_values('Importance', ascending=False).head(20)
    display(importance_df.round(4))


Best-performing model in this run: LightGBM


,Feature,Importance
1,num__num_lab_procedures,834
3,num__num_medications,670
0,num__timeinhospital,489
8,num__age_numeric,366
7,num__number_diagnoses,293
2,num__num_procedures,273
6,num__numberinpatient,259
4,num__number_outpatient,129
105,cat__medical_specialty_Unknown,125
34,cat__payer_code_Unknown,114





- **Logistic Regression**: best when the group wants explainability and a clean academic story.
- **Random Forest**: useful when nonlinear patterns matter but explainability is still manageable.
- **XGBoost / LightGBM**: usually strongest performers on tabular data and best candidates for deployment if metrics are clearly superior.
- For deployment, prefer one **high-performance model** and one **high-interpretability model** so your project shows both practical value and clear explanation.


## Recommended next steps
After reviewing this notebook, the next logical tasks are:
1. tune the best 2 models,
2. select a probability threshold based on recall/precision trade-off,
3. export the fitted deployment pipelines,
4. build the deployment notebook or app interface.
